# Burmese Word Segmentation with a Character-Level CRF

This notebook adapts the character-boundary approach in [mmCRFseg](https://github.com/thuraaungjune/mmCRFseg/blob/main/notebook/mm_word_seg_crf.ipynb). Each character receives a `B` label at the start of a word or `I` otherwise. The existing train and held-out test files contain whitespace-separated `word/POS` tokens; POS tags are used only to recover the gold word boundaries.

## 1. Imports and paths

In [1]:
from pathlib import Path
import pickle
import time
from typing import Sequence

import sklearn_crfsuite
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "dataset").is_dir() and (ROOT_DIR.parent / "dataset").is_dir():
    ROOT_DIR = ROOT_DIR.parent

DATA_DIR = ROOT_DIR / "dataset"
TRAIN_PATH = DATA_DIR / "mypos-ver.3.0.shuf.nopipe.txt"
TEST_PATH = DATA_DIR / "otest.1k.nopipe.txt"
MODEL_PATH = ROOT_DIR / "models" / "mm_word_seg_crf.pkl"

for path in (TRAIN_PATH, TEST_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Required corpus file not found: {path}")

print(f"Project root: {ROOT_DIR}")

Project root: c:\Users\onest\OneDrive\Desktop\Projects\BiLSTM_CRF_Burmese_POS_Tagger


## 2. Load word-tokenized corpora

The parser splits each token at its final slash so a slash inside a word does not corrupt the POS tag. Malformed tokens are reported with their line numbers instead of being silently dropped.

In [2]:
def load_corpus(path: Path) -> list[list[str]]:
    """Load a whitespace-tokenized word/POS corpus as sentences of words."""
    sentences: list[list[str]] = []
    malformed: list[str] = []

    with path.open(encoding="utf-8") as corpus:
        for line_number, line in enumerate(corpus, start=1):
            words: list[str] = []
            for token in line.split():
                word, separator, tag = token.rpartition("/")
                if not separator or not word or not tag:
                    malformed.append(f"line {line_number}: {token!r}")
                    continue
                words.append(word)
            if words:
                sentences.append(words)

    if malformed:
        preview = "; ".join(malformed[:5])
        raise ValueError(f"Found {len(malformed)} malformed tokens in {path}: {preview}")
    if not sentences:
        raise ValueError(f"No usable sentences found in {path}")
    return sentences


train_words = load_corpus(TRAIN_PATH)
test_words = load_corpus(TEST_PATH)
print(f"Training sentences: {len(train_words):,}")
print(f"Test sentences:     {len(test_words):,}")
print("Example gold words:", train_words[0][:12])

Training sentences: 43,196
Test sentences:     1,000
Example gold words: ['၁၉၆၂', 'ခုနှစ်', 'ခန့်မှန်း', 'သန်းခေါင်စာရင်း', 'အရ', 'လူဦးရေ', '၁၁၅၉၃၁', 'ယောက်', 'ရှိ', 'သည်', '။']


## 3. Convert word boundaries to character labels

A `B` marks the first Unicode code point of each word; all remaining code points are `I`. Python indexes Unicode code points, not Burmese grapheme clusters, matching the character-level setup in the reference. The transformation checks that every non-empty word contributes a label.

In [13]:
def prepare_sentence(words: Sequence[str]) -> tuple[list[str], list[str]]:
    """Flatten words into characters and mark the first character of each word."""
    chars: list[str] = []
    labels: list[str] = []
    for word in words:
        if not word:
            raise ValueError("Corpus contains an empty word")
        chars.extend(word)
        labels.extend(["B"] + ["I"] * (len(word) - 1))
    if len(chars) != len(labels):
        raise RuntimeError("Character and label sequence lengths differ")
    return chars, labels


train_sequences = [prepare_sentence(words) for words in train_words]
test_sequences = [prepare_sentence(words) for words in test_words]
print("First sentence characters:", train_sequences[0][0])
print("First sentence labels:    ", train_sequences[0][1])

First sentence characters: ['၁', '၉', '၆', '၂', 'ခ', 'ု', 'န', 'ှ', 'စ', '်', 'ခ', 'န', '်', '့', 'မ', 'ှ', 'န', '်', 'း', 'သ', 'န', '်', 'း', 'ခ', 'ေ', 'ါ', 'င', '်', 'စ', 'ာ', 'ရ', 'င', '်', 'း', 'အ', 'ရ', 'လ', 'ူ', 'ဦ', 'း', 'ရ', 'ေ', '၁', '၁', '၅', '၉', '၃', '၁', 'ယ', 'ေ', 'ာ', 'က', '်', 'ရ', 'ှ', 'ိ', 'သ', 'ည', '်', '။']
First sentence labels:     ['B', 'I', 'I', 'I', 'B', 'I', 'I', 'I', 'I', 'I', 'B', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'B', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'B', 'I', 'B', 'I', 'I', 'I', 'I', 'I', 'B', 'I', 'I', 'I', 'I', 'I', 'B', 'I', 'I', 'I', 'I', 'B', 'I', 'I', 'B', 'I', 'I', 'B']


## 4. Character features

For every character, the feature dictionary contains its identity, nearby characters up to three positions on either side, local n-grams that include the current character, and sentence-boundary indicators. `CRF` consumes one feature dictionary per character.

In [4]:
def character_features(chars: Sequence[str], index: int) -> dict[str, object]:
    """Build local context and character n-gram features at one position."""
    if not 0 <= index < len(chars):
        raise IndexError(f"Character index {index} is outside a sequence of length {len(chars)}")

    features: dict[str, object] = {"bias": 1.0, "char": chars[index]}
    for offset in (-3, -2, -1, 1, 2, 3):
        neighbor = index + offset
        if 0 <= neighbor < len(chars):
            features[f"char[{offset}]"] = chars[neighbor]
        elif neighbor < 0:
            features[f"char[{offset}]"] = "<BOS>"
        else:
            features[f"char[{offset}]"] = "<EOS>"

    for width in (2, 3, 4):
        first_start = max(0, index - width + 1)
        last_start = min(index, len(chars) - width)
        for start in range(first_start, last_start + 1):
            end = start + width
            features[f"ngram[{start - index}:{end - index - 1}]"] = "".join(chars[start:end])

    if index == 0:
        features["BOS"] = True
    if index == len(chars) - 1:
        features["EOS"] = True
    return features


def sentence_features(chars: Sequence[str]) -> list[dict[str, object]]:
    """Create the CRF feature sequence for one sentence."""
    return [character_features(chars, index) for index in range(len(chars))]


example_chars, _ = train_sequences[0]
print(example_chars[:8])
print(sentence_features(example_chars)[0])

['၁', '၉', '၆', '၂', 'ခ', 'ု', 'န', 'ှ']
{'bias': 1.0, 'char': '၁', 'char[-3]': '<BOS>', 'char[-2]': '<BOS>', 'char[-1]': '<BOS>', 'char[1]': '၉', 'char[2]': '၆', 'char[3]': '၂', 'ngram[0:1]': '၁၉', 'ngram[0:2]': '၁၉၆', 'ngram[0:3]': '၁၉၆၂', 'BOS': True}


In [5]:
X_train = [sentence_features(chars) for chars, _ in train_sequences]
y_train = [labels for _, labels in train_sequences]
X_test = [sentence_features(chars) for chars, _ in test_sequences]
y_test = [labels for _, labels in test_sequences]

assert len(X_train) == len(y_train) == len(train_words)
assert len(X_test) == len(y_test) == len(test_words)
assert all(len(features) == len(labels) for features, labels in zip(X_train, y_train))
assert all(len(features) == len(labels) for features, labels in zip(X_test, y_test))
print(f"Training characters: {sum(map(len, y_train)):,}")
print(f"Test characters:     {sum(map(len, y_test)):,}")

Training characters: 2,332,186
Test characters:     56,175


## 5. Train the CRF


In [6]:
crf_model = sklearn_crfsuite.CRF(
    algorithm="lbfgs",
    c1=1.0,
    c2=1e-3,
    max_iterations=100,
    all_possible_transitions=True,
)

started = time.perf_counter()
crf_model.fit(X_train, y_train)
elapsed = time.perf_counter() - started
print(f"Trained in {elapsed:.1f} seconds")

Trained in 104.6 seconds


## 6. Evaluate word-boundary prediction

Boundary precision, recall, and F1 treat `B` as the positive class. Character accuracy measures both boundary and non-boundary labels. All scores are computed on the held-out test corpus.

In [8]:
y_pred = crf_model.predict(X_test)

y_true_flat = [label for sequence in y_test for label in sequence]
y_pred_flat = [label for sequence in y_pred for label in sequence]

if len(y_true_flat) != len(y_pred_flat):
    raise RuntimeError("Gold and predicted character counts differ")

boundary_precision, boundary_recall, boundary_f1, _ = precision_recall_fscore_support(
    y_true_flat,
    y_pred_flat,
    labels=["B"],
    average="binary",
    pos_label="B",
    zero_division=0
)

character_accuracy = accuracy_score(y_true_flat, y_pred_flat)

print(f"Boundary precision: {boundary_precision:.4f}")
print(f"Boundary recall:    {boundary_recall:.4f}")
print(f"Boundary F1:        {boundary_f1:.4f}")
print(f"Character accuracy: {character_accuracy:.4f}")

Boundary precision: 0.9698
Boundary recall:    0.9779
Boundary F1:        0.9738
Character accuracy: 0.9874


## 7. Segment new text

Unspaced text is segmented by the model. Existing whitespace is preserved, and each non-whitespace span is segmented independently. Punctuation is treated like any other character and its spacing follows the training corpus.

In [9]:
def segment_unspaced(text: str, model: sklearn_crfsuite.CRF) -> str:
    """Segment one non-whitespace text span; empty input returns an empty string."""
    chars = list(text)
    if not chars:
        return ""
    labels = model.predict_single(sentence_features(chars))
    if len(labels) != len(chars):
        raise RuntimeError("Prediction length does not match input character count")

    output: list[str] = []
    for char, label in zip(chars, labels):
        if label == "B" and output:
            output.append(" ")
        output.append(char)
    return "".join(output)


def segment_text(text: str, model = None) -> str:
    """Segment unspaced spans while retaining whitespace already in the input."""
    if model is None:
        model = crf_model
    pieces: list[str] = []
    span: list[str] = []
    for char in text:
        if char.isspace():
            if span:
                pieces.append(segment_unspaced("".join(span), model))
                span.clear()
            pieces.append(char)
        else:
            span.append(char)
    if span:
        pieces.append(segment_unspaced("".join(span), model))
    return "".join(pieces)


print(segment_text("ရေပုံမှန်သောက်ပါ"))
print(segment_text("မြန်မာစာ စမ်းသပ်မှု"))
assert segment_text("") == ""
assert segment_text("  ") == "  "

ရေ ပုံမှန် သောက် ပါ
မြန်မာ စာ စမ်းသပ် မှု


In [10]:
for words in test_words[:5]:
    gold = " ".join(words)
    predicted = segment_text("".join(words))
    print("Gold:     ", gold)
    print("Predicted:", predicted)
    print()

Gold:      တစ် ကိုက် ကို ဝမ် ခုနှစ်ထောင် ပါ ။
Predicted: တစ် ကိုက် ကို ဝမ် ခုနှစ်ထောင် ပါ ။

Gold:      မနှစ် က သူ ကျွန်မ ကို သင် ပေး တယ် ။
Predicted: မနှစ် က သူ ကျွန်မ ကို သင် ပေး တယ် ။

Gold:      ကျွန်တော့် ခုံ သွား ရှာ မလို့ ။
Predicted: ကျွန်တော့် ခုံ သွား ရှာ မ လို့ ။

Gold:      အတန်း စ တာ ကြာ ပြီ လား ။
Predicted: အတန်းစတာ ကြာ ပြီ လား ။

Gold:      ဆေး နည်းနည်း စား လိုက် ၊ သုံး လေး ရက် လောက် အနားယူ လိုက် ရင် ပျောက် သွား မှာ ပါ ။
Predicted: ဆေး နည်းနည်းစား လိုက် ၊ သုံး လေး ရက် လောက် အနားယူ လိုက် ရင် ပျောက် သွား မှာ ပါ ။



## 8. Save and reload the model

In [11]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
with MODEL_PATH.open("wb") as model_file:
    pickle.dump(crf_model, model_file)
print(f"Saved model to {MODEL_PATH}")

with MODEL_PATH.open("rb") as model_file:
    loaded_model = pickle.load(model_file)
print(segment_text("ရေပုံမှန်သောက်ပါ", model=loaded_model))

Saved model to c:\Users\onest\OneDrive\Desktop\Projects\BiLSTM_CRF_Burmese_POS_Tagger\models\mm_word_seg_crf.pkl
ရေ ပုံမှန် သောက် ပါ


## Notes and limitations

- Scores are character-level boundary metrics; they are not sentence-level exact-match scores.
- The tagger learns the tokenization conventions and coverage of the available corpus. Domain shifts, unseen words, punctuation conventions, and inconsistent annotations can affect results.
- The training feature lists can use substantial memory. If needed for experimentation, reduce the training sentences explicitly, but do not report that subset run as the full-corpus result.

## Reference

- myCRFSeg: Word Segmentation Tool for Myanmar Language using Binary Tagging based Conditional Random Fields, Thura Aung, July 2022, GitHub Link: https://github.com/ThuraAung1601/mmCRFseg  